# KAN Hyperparameter Optimization

Optuna searches the five selected KAN hyperparameters and saves every completed trial in one timestamped results directory.

In [4]:
import os
import sys
from datetime import datetime

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns
import infinite
reload(plotting)
reload(pinns)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

## Optuna Search Configuration

In [5]:
import optuna

# Search space from the architecture-matching study.
KAN_SEARCH_SPACE = {
    "hidden_layers": [1, 2, 3],
    "hidden_units": [15, 25, 35],
    "grid_size": [3, 5, 7],
    "spline_order": [2, 3, 4],
    "learning_rate": [1e-4, 1e-3, 1e-2],
}

N_TRIALS = 10
ADAM_ITERS = 1
LBFGS_ITERS = 1

# One directory contains the CSV summary and all saved trial models.
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
results_dir = f"results_kan_optuna_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

print(f"Results will be saved to: {results_dir}")
print(f"Optuna trials: {N_TRIALS}")

Results will be saved to: results_kan_optuna_2026-09-14_19-12-22
Optuna trials: 10


## Objective Function

In [6]:
def objective(trial):
    """Run one KAN training configuration and return mean global error."""
    config = {
        "hidden_layers": trial.suggest_categorical(
            "hidden_layers",
            KAN_SEARCH_SPACE["hidden_layers"],
        ),
        "hidden_units": trial.suggest_categorical(
            "hidden_units",
            KAN_SEARCH_SPACE["hidden_units"],
        ),
        "grid_size": trial.suggest_categorical(
            "grid_size",
            KAN_SEARCH_SPACE["grid_size"],
        ),
        "spline_order": trial.suggest_categorical(
            "spline_order",
            KAN_SEARCH_SPACE["spline_order"],
        ),
        "learning_rate": trial.suggest_categorical(
            "learning_rate",
            KAN_SEARCH_SPACE["learning_rate"],
        ),
    }

    print(
        f"\n--- Trial {trial.number}: "
        f"L={config['hidden_layers']}, "
        f"N={config['hidden_units']}, "
        f"grid={config['grid_size']}, "
        f"order={config['spline_order']}, "
        f"lr={config['learning_rate']:.0e} ---"
    )

    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="KAN",
            hidden_layers=config["hidden_layers"],
            hidden_units=config["hidden_units"],
            grid_size=config["grid_size"],
            spline_order=config["spline_order"],
            adam_lr=config["learning_rate"],
            device=device,
            adam_iters=ADAM_ITERS,
            lbfgs_iters=LBFGS_ITERS,
            results_dir=results_dir,
        )
    except Exception as error:
        print(f"Trial {trial.number} failed: {error}")
        raise optuna.exceptions.TrialPruned()

    mean_global_error = 0.5 * (err_u + err_k)
    trial.set_user_attr("err_u", float(err_u))
    trial.set_user_attr("err_k", float(err_k))
    trial.set_user_attr("compute_time_sec", float(compute_time))

    print(
        f"Success! Time: {compute_time:.2f}s | "
        f"Err U: {err_u:.3e} | Err K: {err_k:.3e} | "
        f"Mean error: {mean_global_error:.3e}"
    )
    return mean_global_error

## Run Optimization

In [7]:
sampler = optuna.samplers.TPESampler(seed=1)
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    study_name=f"kan_infinite_domain_{timestamp}",
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    catch=(RuntimeError, ValueError),
)

print("\n========================================")
print("BEST KAN CONFIGURATION")
print("========================================")
print(f"Mean global error: {study.best_value:.6e}")
print("Parameters:")
for name, value in study.best_params.items():
    print(f"  {name}: {value}")

[I 2026-09-14 19:12:33,346] A new study created in memory with name: kan_infinite_domain_2026-09-14_19-12-22



--- Trial 0: L=2, N=15, grid=7, order=4, lr=1e-03 ---


/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[I 2026-09-14 19:12:39,291] Trial 0 finished with value: 0.9997310367414082 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'grid_size': 7, 'spline_order': 4, 'learning_rate': 0.001}. Best is trial 0 with value: 0.9997310367414082.



[KAN] L=2, N=15 | Params: 7,020 | Mean Err: 9.997e-01 | Saved to 'results_kan_optuna_2026-09-14_19-12-22/'.
Success! Time: 5.93s | Err U: 6.646e-02 | Err K: 1.933e+00 | Mean error: 9.997e-01

--- Trial 1: L=1, N=35, grid=3, order=3, lr=1e-02 ---


[I 2026-09-14 19:12:41,366] Trial 1 finished with value: 0.6900206011070484 and parameters: {'hidden_layers': 1, 'hidden_units': 35, 'grid_size': 3, 'spline_order': 3, 'learning_rate': 0.01}. Best is trial 1 with value: 0.6900206011070484.


...


[KAN] L=3, N=25 | Params: 26,500 | Mean Err: 1.033e+00 | Saved to 'results_kan_optuna_2026-09-14_19-12-22/'.
Success! Time: 6.64s | Err U: 6.678e-02 | Err K: 1.999e+00 | Mean error: 1.033e+00

--- Trial 8: L=3, N=25, grid=5, order=4, lr=1e-02 ---


[I 2026-09-14 19:14:00,854] Trial 8 finished with value: 0.7897756652477165 and parameters: {'hidden_layers': 3, 'hidden_units': 25, 'grid_size': 5, 'spline_order': 4, 'learning_rate': 0.01}. Best is trial 1 with value: 0.6900206011070484.



[KAN] L=3, N=25 | Params: 29,150 | Mean Err: 7.898e-01 | Saved to 'results_kan_optuna_2026-09-14_19-12-22/'.
Success! Time: 19.78s | Err U: 5.373e-02 | Err K: 1.526e+00 | Mean error: 7.898e-01

--- Trial 9: L=2, N=15, grid=3, order=4, lr=1e-04 ---


[I 2026-09-14 19:14:02,376] Trial 9 finished with value: 0.8729080134522147 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'grid_size': 3, 'spline_order': 4, 'learning_rate': 0.0001}. Best is trial 1 with value: 0.6900206011070484.



[KAN] L=2, N=15 | Params: 4,860 | Mean Err: 8.729e-01 | Saved to 'results_kan_optuna_2026-09-14_19-12-22/'.
Success! Time: 1.51s | Err U: 8.551e-02 | Err K: 1.660e+00 | Mean error: 8.729e-01

BEST KAN CONFIGURATION
Mean global error: 6.900206e-01
Parameters:
  hidden_layers: 1
  hidden_units: 35
  grid_size: 3
  spline_order: 3
  learning_rate: 0.01
